# Women in the Encyclopedia of Australian Science (EOAS)

The [Encyclopedia of Australian Science and Innovation](https://www.eoas.info) (EOAS) is a
biographical register of people, organisations and resources from the history of Australian
science. This register is saved locally in RO-Crate.

This notebook takes us through a set of research questions about the **women** recorded
in the collection, using `crategraph` for all of the exploratory work, subsequently handling data in pandas/Plotly 
at the final stage.

**Research questions**

1. Who is the first woman recorded in this collection: a pioneer, or an artefact of sparse early data?
2. Did these women come from educated families?
3. When did the *boom* of female scientists happen?
4. Which fields attracted women most?
5. Are there demographic patterns among them?
6. What is the highest share of women vs men recorded at any point in time?
7. Can we see era trends in their choice of fields?

The notebook is organised into four parts: **§1 a clean dataset**, **§2 timing**, **§3 fields**,
and **§4 backgrounds**.

## 1. Building a community of women

Every later question depends on one thing: a trustworthy set of "the women in EOAS". First of all,
we need to understand how gender is recorded, and rule out a couple of data-quality
traps. Let's load the crate first.

In [ ]:
from crategraph import Crate

crate = Crate("data/ohrm/EOASI2022-ro-crate")
crate

### How is gender recorded?

We ask crategraph to count the values of the `gender` property directly with `entity_counts`.

In [ ]:
crate.entity_counts("gender")

We build the set with `annotate_entities` (to add a derived `is_female` flag) followed by
`where` (to keep only the matches).

In [ ]:
fem = crate.annotate_entities(
    is_female=lambda e: e.get("gender") in ("F","f")
).where(is_female=True)

### Data-quality check: do archivists appear in this set?

EOAS records the archivists who *prepared* entries. They appear as the target of a `preparedBy`
relationship. If any archivist also carried a `gender` value, they would appear in our
dataset (however that's not desired). We test this directly: flag anything with an **incoming** `preparedBy` edge as an
archivist, and see whether removing them changes the count.

In [ ]:
fem_archivist = crate.annotate_entities(
    is_female=lambda e: e.get("gender") in ("F", "f"),
    is_archivist=lambda e: e.has("preparedBy", direction="in"),
).where(is_female=True).where(is_archivist=True)

len(fem_archivist)

We *verified* that there're no archivists that carry a female gender value, so we can consider our `fem` dataset clean. 

### Data-quality check: `function` vs `x_efunction`

EOAS stores a person's profession in two properties, `function` and `x_efunction`. They mostly
agree, but not entirely. We compare how many of the dataset have each one populated, so we know
which to trust for the field analysis in §3.

In [ ]:
fem_with_function = fem.annotate_entities(
    has_it=lambda e: bool(e.get("function"))
).where(has_it=True)

fem_with_xefunction = fem.annotate_entities(
    has_it=lambda e: bool(e.get("x_efunction"))
).where(has_it=True)


fem_no_function = fem.annotate_entities(
      has_it=lambda e: bool(e.get("function"))
  ).where(has_it=False)

len(fem_with_function), len(fem_with_xefunction), len(fem_no_function)

`function` is populated for more of the dataset than `x_efunction`, and is effectively a superset
of it, so **we use `function`** as the profession field throughout. 
23 women have no proffession listed.
A quick look at the most common values:

In [ ]:
fem.entity_counts("function")

We can inspect the mismatch between `function` and `x_efunction` by creating a separate dataset, and then visualising it in Pandas:

In [ ]:
mismatch = fem.annotate_entities(
      differs=lambda e: e.get("function") != e.get("x_efunction")
  ).where(differs=True)

len(mismatch)

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(mismatch.entity_records(columns=["name", "function", "x_efunction"]))[20:29]

After confirming that it's reasonable to use `function` as it essentially covers `x_efunction`, we can proceed with creating a dataset of proffessions. Each woman can holf more than one role, which we will explore next:

In [ ]:
professions = (
      pd.DataFrame(fem.entity_records(columns=["function"]))["function"]
      .dropna()
      .str.split(", ")     # "Nurse, Nurse educator" -> ["Nurse", "Nurse educator"]
      .explode()           # one row per individual profession
      .value_counts()      # count each on its own
  )

In [ ]:
import plotly.express as px

top_prof = professions.head(20).sort_values()

fig = px.bar(
      x=top_prof.values,
      y=top_prof.index,
      orientation="h",
      title="Top 20 professions among women in EOAS",
      labels={"x": "Number", "y": "Profession"}
  )
fig.update_traces(marker_color="#008080")
fig.update_layout(
      template="plotly_white",
      height=400,                              
      margin=dict(l=10, r=30, t=50, b=10),
  )
fig.show()

In [ ]:
from IPython.display import Markdown
Markdown(
    f"**{len(fem.subtract(fem_no_function))}** women hold "f"**{professions.sum()}** roles across "f"**{len(professions)}** distinct professions."
        )

To find out the most unpopular professions in our dataset, we can use the following approach:

In [ ]:
lowest = professions.min()
unpopular = professions[professions == lowest]
len(unpopular)  

In [ ]:
list(unpopular.index)

There's 131 profession that has been mentiones in relation to only one woman in this dataset.

## 2. Timing — when did these women live?

`startDate` on a person is their birth date, but it's stored as a messy string. `convert_dates()` parses it for us and adds a clean `year`. We can now answer the question about the first woman in this dataset:

In [ ]:
fem_dates = fem.convert_dates(start="startDate", report=False)
fem_birth = sorted(
      (e for e in fem_dates.entities if e.properties.get("year")),
      key=lambda e: e.properties["year"],
  )

### The first woman recorded

Get the first entry in chronologically sorted dataset:

In [ ]:
fem_birth[0].properties

Pauline de Courcelles Knip was a natural history artist. This earliest record sits well before the cluster of later ones: it's a starting point, and not necesserily a proof of a profession pioneer.

### When did the boom happen?

Counting births by year shows when women start appearing in numbers.

In [ ]:
fig = px.histogram(
    years, x="year", nbins=30,
    title="Women in EOAS by birth year",
)
fig.update_traces(marker_color="#008080")
fig.update_layout(template="plotly_white", height=400, margin=dict(l=10, r=30, t=50, b=10))
fig.show()

### Women vs men over time

To see the *share* of women, we build the men's dataset the same way and compare births per decade.

In [ ]:
men = crate.annotate_entities(is_male=lambda e: e.get("gender") in ("M", "m")).where(is_male=True)
mend = men.convert_dates(report=False)

years["decade"] = (years["year"] // 10) * 10

myears = pd.DataFrame(mend.entity_records(columns=["year"])).dropna()
myears["year"] = myears["year"].astype(int)
myears["decade"] = (myears["year"] // 10) * 10

w = years["decade"].value_counts()
m = myears["decade"].value_counts()
share = (w / (w + m) * 100).dropna().sort_index()
share

In [ ]:
fig = px.line(
    x=share.index, y=share.values, markers=True,
    title="Share of women among dated EOAS people, by decade",
    labels={"x": "decade", "y": "% women"},
)
fig.update_traces(line_color="#008080")
fig.update_layout(template="plotly_white", height=400, margin=dict(l=10, r=30, t=50, b=10))
fig.show()

## 3. Era trends in fields

Do the fields women enter change over time? We join each woman's profession to her decade. Because a woman can hold several roles, we split `function` again and keep one row per role.

In [ ]:
# one row per (woman, profession) with her decade
era = pd.DataFrame(femd.entity_records(columns=["function", "year"])).dropna()
era["year"] = era["year"].astype(int)
era["decade"] = (era["year"] // 10) * 10
era = era.assign(function=era["function"].str.split(", ")).explode("function")
len(era)

In [ ]:
# how the top 5 professions rise and fall across decades
top5 = era["function"].value_counts().head(5).index
sub = era[era["function"].isin(top5)]
counts = sub.groupby(["decade", "function"]).size().reset_index(name="count")

fig = px.line(
    counts, x="decade", y="count", color="function", markers=True,
    title="Top 5 professions of women over time",
)
fig.update_layout(template="plotly_white", height=450, margin=dict(l=10, r=30, t=50, b=10))
fig.show()

## 4. Backgrounds

### Did they come from educated families?

We can e Let's count how many of those exist in our dataset before trying to answer.

In [ ]:
fem.relationship_types

In [ ]:
for r in ["Parent", "Child", "Sibling", "Related"]:
    print(r, len(fem.select(relationship_types=rt).relationships))

The family links are very sparse: only a handful across 894 women, hence this dataset can't really tell us whether these women came from educated families. We note it as a limitation rather than forcing an answer.

### Demographic patterns

Where were these women born? `birthState` and `nationality` are well populated, so we can count them directly with `entity_counts`.

In [ ]:
fem.entity_counts("birthState")[:10]

In [ ]:
fem.entity_counts("nationality")[:10]

In [ ]:
states = pd.DataFrame(fem.entity_counts("birthState")[:10]).sort_values("count")

fig = px.bar(
    states, x="count", y="birthState", orientation="h",
    title="Where women in EOAS were born (top 10)",
)
fig.update_traces(marker_color="#008080")
fig.update_layout(template="plotly_white", height=400, margin=dict(l=10, r=30, t=50, b=10))
fig.show()

## Conclusion

Across the seven questions: we built a verified dataset of **894 women**, found that the
"first" woman reflects where the records start more than a true pioneer, saw when women began
appearing in numbers and how their share shifted over time, watched the mix of fields change
by decade, and looked at where they were born, while noting the family data is too sparse to
say much about their backgrounds.